In [3]:
import requests
import pandas as pd
import json
import os
import numpy as np
from time import sleep

pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

import warnings
warnings.filterwarnings('ignore')

In [4]:
precos_teto_suno_dividendos = {
    "WIZC3": 10.00, "BBSE3": 35.50, "BBAS3": 25.00, "UNIP6": 70.00, "SEER3": 14.00, "VALE3": 75.00,
    "PETR4": 34.00, "AXIA6": 43.80, "TUPY3": 21.00, "AGRO3": 27.50, "EGIE3": 28.60, "ITSA4": 9.50,
}

precos_teto_suno_valor = {"VAMO3": 10.90, "B3SA3": 17.00, "KLBN11": 25.00, "TTEN3": 14.70, "PRIO3": 62.75, "BRBI11": 18.00,
    "PNVL3": 12.00, "SIMH3": 10.00, "GMAT3": 7.12, "TIMS3": 18.60, "VIVA3": 25.00, "EZTC3": 13.29,
    "BRKM5": 999,
}



carteira_PM = {
    "ABEV3": 10.00, "B3SA3": 11.08, "BBAS3": 21.96, "BBSE3": 32.00,
    "EGIE3": 28, "FLRY3": 15.60, "HYPE3": 29.31, "ITSA4": 9.41,
    "KLBN11": 18.58, "LEVE3": 33.92, "PETR4": 30.06, "TAEE11": 38.28,
    "UNIP6": 50.99, "VALE3": 58.13, "RADL3":25
}

# Valuation Gemini - Fevereiro/Março 2026
# Metodologia: Crescimento de Lucro (CAGR) + Múltiplos Setoriais
# Premissas: Juros 9% a.a. | Margem de Segurança: 30%

valuation_gemini_9pct = {
    "VALE3": 75.60, "PETR4": 35.00, "BBAS3": 40.95, "BBSE3": 32.20,
    "PRIO3": 50.40, "ITSA4": 12.04, "EGIE3": 37.80, "TAEE11": 30.45,
    "ABEV3": 12.25, "LEVE3": 35.70, "HYPE3": 30.80, "FLRY3": 16.45,
    "KLBN11": 21.00, "UNIP6": 71.40, "B3SA3": 11.34, "TTEN3": 13.65,
    "VAMO3": 9.24, "SIMH3": 8.26, "GMAT3": 8.05, "EZTC3": 18.20,
    "TUPY3": 26.95, "AGRO3": 23.80, "TIMS3": 15.96, "VIVA3": 23.10,
    "WIZC3": 8.54, "BRKM5": 25.20, "PNVL3": 10.85, "SEER3": 8.96,
    "BRBI11": 18.90, "AXIA6": 33.60, "DEXP4":10.15
}

valuation_perene_extra = {
    # Saneamento
    "SBSP3": 80.50, "SAPR11": 26.60, "CSMG3": 19.60,
    # Transmissão e Agro
    "TRPL4": 25.20, "SLCE3": 18.20, 
    # Logística e Adições de Valor
    "STBP3": 12.95, "SUZB3": 54.60, "MDIA3": 31.50
}

precos_teto = {}
for k, v in carteira_PM.items():
    precos_teto[k] = precos_teto_suno_dividendos.get(k, v)

# precos_teto = precos_teto_suno

# precos_teto.update(valuation_perene_extra)



In [5]:
if not os.path.exists("dbJson"):
    os.makedirs("dbJson")

def atualizarDados(ativo):
    url = f"https://storage.googleapis.com/api-cdn-eaglesystem/api/{ativo.upper()}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return f"{ativo}: erro HTTP {response.status_code}"
        
        # Validação de conteúdo JSON para evitar erro 'Expecting value'
        content = response.text.strip()
        if not (content.startswith('{') or content.startswith('[')):
            return f"{ativo}: Erro - Conteúdo não é JSON válido"

        dados = response.json()
        with open(f"dbJson/{ativo}.json", "w", encoding="utf-8") as arq:
            json.dump(dados, arq, indent=2)
        return f"{ativo}: atualizado"
    except Exception as e:
        return f"{ativo}: erro -> {e}"

for ativo in precos_teto.keys():
    print(atualizarDados(ativo))

ABEV3: atualizado
B3SA3: atualizado
BBAS3: atualizado
BBSE3: atualizado
EGIE3: atualizado
FLRY3: atualizado
HYPE3: atualizado
ITSA4: atualizado
KLBN11: atualizado
LEVE3: atualizado
PETR4: atualizado
TAEE11: atualizado
UNIP6: atualizado
VALE3: atualizado
RADL3: atualizado


In [6]:
# ---------------------------------------------------
# CRIAR DATAFRAME
# ---------------------------------------------------

def dataFrameUnico(ativo):

    with open(f"dbJson/{ativo}.json", "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    preco_atual = dados["asset"]["close"]

    linhas = []

    for serie in dados["series"]:

        vencimento = serie.get("due_date")
        dias = serie.get("days_to_maturity")

        for strike in serie["strikes"]:

            strike_price = strike["strike"]

            # CALL
            if strike["call"]:

                c = strike["call"]

                linhas.append({
                    "ativo": ativo,
                    "tipo": "CALL",
                    "vencimento": vencimento,
                    "dias": dias,
                    "strike": strike_price,
                    "symbol": c["symbol"],
                    "bid": c["bid"],
                    "ask": c["ask"],
                    "volume": c["volume"],
                    "delta": c["bs"]["delta"],
                    "theta": c["bs"]["theta"],
                    "vol": c["bs"]["volatility"],
                    "poe": c["bs"]["poe"],
                    "preco_atual": preco_atual
                })

            # PUT
            if strike["put"]:

                p = strike["put"]

                linhas.append({
                    "ativo": ativo,
                    "tipo": "PUT",
                    "vencimento": vencimento,
                    "dias": dias,
                    "strike": strike_price,
                    "symbol": p["symbol"],
                    "bid": p["bid"],
                    "ask": p["ask"],
                    "volume": p["volume"],
                    "delta": p["bs"]["delta"],
                    "theta": p["bs"]["theta"],
                    "vol": p["bs"]["volatility"],
                    "poe": p["bs"]["poe"],
                    "preco_atual": preco_atual
                })

    df = pd.DataFrame(linhas)

    return df





# ---------------------------------------------------
# CRIAR DATAFRAME GERAL
# ---------------------------------------------------

todos = []

for ativo, preco_teto in precos_teto.items():

    try:

        df = dataFrameUnico(ativo)

        df["preco_teto"] = preco_teto

        todos.append(df)

    except Exception as e:

        print(f"Erro em {ativo}: {e}")
        pass


df_final = pd.concat(todos, ignore_index=True)

In [7]:
# ------------------------------------------
# FILTRO PARA VENDA DE PUT
# ------------------------------------------
puts = df_final[df_final["tipo"] == "PUT"]

puts["retorno"] = (puts["bid"]/ (puts["strike"] - puts['bid']))  # Retorno ROR
puts["retorno_mes"] = puts["retorno"] * (30 / puts["dias"])

puts['retorno'] = round(puts['retorno'] * 100,2)
puts['retorno_mes'] = round(puts['retorno_mes'] * 100,2)


puts["dist_strike"] = round((puts["strike"] / puts["preco_atual"] - 1) * 100,2)

# ------------------------------------------
# APORTE
# ------------------------------------------

aporte = 10000

puts['cotas'] = np.floor((aporte / puts['strike'])/100) * 100
puts['premio X cotas'] = puts['cotas'] * puts['bid']

# ------------------------------------------
# BLACK scholes
# ------------------------------------------

import numpy as np
from scipy.stats import norm

r = 0.1475  # taxa livre de risco

T = puts['dias'] / 252

# volatilidade em decimal
sigma = puts['vol'] / 100

d1 = (
    np.log(puts['preco_atual'] / puts['strike']) +
    (r + sigma**2 / 2) * T
) / (sigma * np.sqrt(T))

d2 = d1 - sigma * np.sqrt(T)

puts['black_scholes'] = round((
    puts['strike'] * np.exp(-r * T) * norm.cdf(-d2)
    - puts['preco_atual'] * norm.cdf(-d1)
),2)

puts['desvio_bs'] = round( puts['bid'] - puts['black_scholes'],2)




In [8]:
puts

,ativo,tipo,vencimento,dias,strike,symbol,bid,ask,volume,delta,theta,vol,poe,preco_atual,preco_teto,retorno,retorno_mes,dist_strike,cotas,premio X cotas,black_scholes,desvio_bs
1,ABEV3,PUT,2026-04-10,4,11.67,ABEVP116W2,0.00,0.0,0,0.000000,0.000000e+00,0.000,0.00,15.40,10.0,0.00,0.00,-24.22,800.0,0.0,0.00,0.00
3,ABEV3,PUT,2026-04-10,4,12.17,ABEVP121W2,0.00,0.0,0,0.000000,0.000000e+00,0.000,0.00,15.40,10.0,0.00,0.00,-20.97,800.0,0.0,0.00,0.00
5,ABEV3,PUT,2026-04-10,4,12.67,ABEVP126W2,0.00,0.0,0,0.000000,0.000000e+00,0.000,0.00,15.40,10.0,0.00,0.00,-17.73,700.0,0.0,0.00,0.00
7,ABEV3,PUT,2026-04-10,4,13.17,ABEVP131W2,0.00,0.0,0,0.000000,-6.019528e-08,0.000,0.00,15.40,10.0,0.00,0.00,-14.48,700.0,0.0,0.00,0.00
9,ABEV3,PUT,2026-04-10,4,13.67,ABEVP136W2,0.00,0.0,500,-0.000048,-1.142304e-05,50.568,0.01,15.40,10.0,0.00,0.00,-11.23,700.0,0.0,0.01,-0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17549,RADL3,PUT,2027-02-19,219,36.19,RADLN361,0.00,0.0,0,-0.806224,1.179335e-02,0.000,88.96,21.83,25.0,0.00,0.00,65.78,200.0,0.0,10.01,-10.01
17551,RADL3,PUT,2027-04-16,258,20.27,RADLP209,0.01,0.0,0,-0.219052,-1.279855e-03,31.755,35.03,21.83,25.0,0.05,0.01,-7.15,400.0,4.0,0.89,-0.88
17553,RADL3,PUT,2027-08-20,346,19.46,RADLT200,0.01,0.0,0,-0.175623,-7.279808e-04,34.038,31.58,21.83,25.0,0.05,0.00,-10.86,500.0,5.0,0.89,-0.88
17555,RADL3,PUT,2027-12-17,427,22.00,RADLX220,0.00,0.0,0,-0.230233,2.078554e-04,37.696,40.70,21.83,25.0,0.00,0.00,0.78,400.0,0.0,1.87,-1.87


In [9]:
filtro = puts[

    (puts['dias'].between(1, 90)) &
    (puts['dist_strike'] <= -10) &
    (puts['retorno_mes'] >= 1) & # maior que o cdi liquido mensal
    # (puts['retorno'] >= 1) &
    (puts['delta'] >= -0.30 )&
    (puts['poe'] <= 30) &
    (puts['strike'] <= puts['preco_teto'] * 1.05) &
    (puts['bid'] >= 0.5) &
    # (puts['premio X cotas'] >= 80) &
    
    (puts['volume'] > 0)

].copy()

filtro['rank_retorno_mes'] = filtro['retorno_mes'].rank(ascending=False)
filtro['rank_dias'] = filtro['dias'].rank(ascending=True)
filtro['rank_dist_strike'] = filtro['dist_strike'].rank(ascending=True)
filtro['rank_poe'] = filtro['poe'].rank(ascending=True)
filtro['rank_premio X cotas'] = filtro['premio X cotas'].rank(ascending=False)

filtro['score'] = (
    filtro['rank_dist_strike'] * 5 +
    filtro['rank_poe'] * 3 +
    filtro['rank_retorno_mes'] * 4 +
    filtro['rank_premio X cotas'] * 3.5 +
    filtro['rank_dias'] * 3
)

filtro[[
    'ativo', 'symbol', 'tipo',  'preco_atual', 'strike', 'dist_strike', 'bid','black_scholes','desvio_bs' ,'ask',
    'volume', 'delta', 'theta', 'vol', 'poe',  'preco_teto',
    'retorno', 'retorno_mes', 'vencimento',  'dias', 'cotas', 'premio X cotas','score'
]].sort_values('score', ascending=True)

,ativo,symbol,tipo,preco_atual,strike,dist_strike,bid,black_scholes,desvio_bs,ask,volume,delta,theta,vol,poe,preco_teto,retorno,retorno_mes,vencimento,dias,cotas,premio X cotas,score


In [10]:
puts["retorno"] = (puts["bid"]/ (puts["strike"] - puts['bid']))  # Retorno ROR
puts["retorno_anual"] = puts["retorno"] * (365 / puts["dias"])

puts['retorno'] = round(puts['retorno'] * 100,2)
puts['retorno_anual'] = round(puts['retorno_anual'] * 100,2)


In [11]:
# PUT LONGA

puts[
    # (puts['dias'].between(90,720)) &
    (puts['volume'] > 0 ) & 
    # (puts['retorno_anual'] >= 14)
    (puts['black_scholes'] >= 2) &
    # (puts['bid'] == 0) &
    (puts['dist_strike'] <= -5)
][[
'ativo', 'symbol', 'tipo',  'preco_atual', 'strike', 'dist_strike', 'bid','black_scholes','desvio_bs' ,'ask',
    'volume', 'delta', 'theta', 'vol', 'poe',  'preco_teto',
    'retorno', 'retorno_anual', 'vencimento',  'dias', 'cotas', 'premio X cotas'
]].sort_values('retorno_anual', ascending= False)



,ativo,symbol,tipo,preco_atual,strike,dist_strike,bid,black_scholes,desvio_bs,ask,volume,delta,theta,vol,poe,preco_teto,retorno,retorno_anual,vencimento,dias,cotas,premio X cotas
12875,PETR4,PETRN43,PUT,48.10,43.50,-9.56,0.95,2.50,-1.55,0.00,400,-0.211218,-0.004320,39.078,33.42,34.0,2.23,3.72,2027-02-19,219,200.0,190.0
12897,PETR4,PETRO48,PUT,48.10,45.00,-6.44,0.90,2.86,-1.96,0.00,500,-0.233395,-0.003530,37.917,36.80,34.0,2.04,3.12,2027-03-19,239,200.0,180.0
13105,PETR4,PETRN429,PUT,48.10,43.00,-10.60,1.05,2.39,-1.34,0.00,106400,-0.162084,-0.000510,36.225,33.06,34.0,2.50,1.94,2028-02-18,470,200.0,210.0
12047,PETR4,PETRT452,PUT,48.10,45.23,-5.97,0.01,2.07,-2.06,0.00,100,-0.273700,-0.011504,37.776,36.16,34.0,0.02,0.08,2026-08-21,96,200.0,2.0
13107,PETR4,PETRN440,PUT,48.10,44.00,-8.52,0.02,2.60,-2.58,0.00,45400,-0.172597,-0.000344,36.107,34.59,34.0,0.05,0.04,2028-02-18,470,200.0,4.0
12537,PETR4,PETRV502,PUT,48.10,44.42,-7.65,0.00,2.41,-2.41,2.55,8400,-0.245301,-0.008121,40.213,34.57,34.0,0.00,0.00,2026-10-16,134,200.0,0.0
12735,PETR4,PETRX447,PUT,48.10,44.72,-7.03,0.00,2.97,-2.97,0.00,100,-0.242905,-0.005694,41.404,35.94,34.0,0.00,0.00,2026-12-18,178,200.0,0.0
13053,PETR4,PETRM430,PUT,48.10,43.00,-10.60,0.00,2.38,-2.38,0.00,400,-0.165121,-0.000667,36.199,33.08,34.0,0.00,0.00,2028-01-21,450,200.0,0.0
13057,PETR4,PETRM440,PUT,48.10,44.00,-8.52,0.00,2.58,-2.58,0.00,400,-0.175998,-0.000501,35.997,34.65,34.0,0.00,0.00,2028-01-21,450,200.0,0.0
15231,VALE3,VALER780W2,PUT,83.69,78.00,-6.80,0.00,3.24,-3.24,0.00,200,-0.203049,-0.021888,47.027,24.09,75.0,0.00,0.00,2026-06-12,46,100.0,0.0


# CALLS

In [12]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# ------------------------------------------
# FILTRAR CALLS
# ------------------------------------------

calls = df_final[df_final["tipo"] == "CALL"].copy()

# remover dados inválidos
calls = calls[
    (calls["preco_atual"] > 0) &
    (calls["strike"] > 0) &
    (calls["dias"] > 0) &
    (calls["vol"] > 0)
].copy()

# ------------------------------------------
# DISTÂNCIA DO STRIKE (%)
# ------------------------------------------

calls["dist_strike"] = (
    (calls["strike"] / calls["preco_atual"] - 1) * 100
).round(2)

# ------------------------------------------
# BLACK SCHOLES
# ------------------------------------------

r = 0.10

T = calls["dias"] / 252
sigma = calls["vol"] / 100

# evitar divisão por zero
T = T.clip(lower=0.0001)
sigma = sigma.clip(lower=0.0001)

d1 = (
    np.log(calls["preco_atual"] / calls["strike"]) +
    (r + sigma**2 / 2) * T
) / (sigma * np.sqrt(T))

d2 = d1 - sigma * np.sqrt(T)

calls["black_scholes"] = (
    calls["preco_atual"] * norm.cdf(d1) -
    calls["strike"] * np.exp(-r * T) * norm.cdf(d2)
)

calls["black_scholes"] = calls["black_scholes"].round(2)

# ------------------------------------------
# DESVIO DO PREÇO TEÓRICO
# ------------------------------------------

calls["desvio_bs"] = (
    calls["black_scholes"] - calls["ask"]
).round(2)

calls["desvio_pct"] = (
    (calls["black_scholes"] - calls["ask"]) /
    calls["black_scholes"]
) * 100

calls["desvio_pct"] = calls["desvio_pct"].round(2)

# ------------------------------------------
# PROBABILIDADE ITM
# ------------------------------------------

calls["prob_itm"] = (norm.cdf(d2) * 100).round(2)

# ------------------------------------------
# RETORNO DA CALL COBERTA
# ------------------------------------------

calls["retorno"] = (
    calls["bid"] / calls["preco_atual"]
)

calls["retorno_anual"] = (
    calls["retorno"] * (365 / calls["dias"])
)

calls["retorno"] = (calls["retorno"] * 100).round(2)
calls["retorno_anual"] = (calls["retorno_anual"] * 100).round(2)


In [13]:
# ------------------------------------------
# FILTRO DO ATIVO
# ------------------------------------------

filtro_call = calls[
    # (calls['ativo'] == 'HYPE3') &
    (calls['ask'].between(0.01,0.1)) &
    # (calls['preco_teto'] <= calls['strike']) &
    (calls['dist_strike'] >= 0) &
    # (calls['dias'].between(1,90)) &
    (calls['retorno_anual'] >= 6) &
    (calls['volume'] > 0)
].copy()

# ------------------------------------------
# RANKING
# ------------------------------------------

filtro_call['rank_retorno'] = filtro_call['retorno_anual'].rank(
    ascending=False)
filtro_call['rank_dist'] = filtro_call['dist_strike'].rank(ascending=False)
filtro_call['rank_prob'] = filtro_call['prob_itm'].rank(ascending=True)
filtro_call['rank_dias'] = filtro_call['dias'].rank(ascending=True)
filtro_call['rank_vol'] = filtro_call['volume'].rank(ascending=False)

# ------------------------------------------
# SCORE FINAL
# ------------------------------------------

filtro_call['score'] = (
    filtro_call['rank_retorno'] * 3 +
    filtro_call['rank_dist'] * 2 +
    filtro_call['rank_prob'] * 3 +
    filtro_call['rank_dias'] * 1 +
    filtro_call['rank_vol'] * 1
)

# ------------------------------------------
# RESULTADO
# ------------------------------------------

resultado = filtro_call[[
    'ativo',
    'symbol',
    'tipo',
    'bid',
    'ask',
    'preco_atual',
    'strike',
    'dist_strike',
    'prob_itm',
    'black_scholes',
    'desvio_bs',
    'desvio_pct',
    'dias',
    'vencimento',
    'preco_teto',
    'retorno',
    'retorno_anual',
    'score',
]].sort_values('score', ascending=True)

resultado

,ativo,symbol,tipo,bid,ask,preco_atual,strike,dist_strike,prob_itm,black_scholes,desvio_bs,desvio_pct,dias,vencimento,preco_teto,retorno,retorno_anual,score
3140,BBAS3,BBASD262,CALL,0.05,0.08,23.43,26.02,11.05,7.63,0.07,-0.01,-14.29,9,2026-04-17,25.00,0.21,8.65,17.5
1690,B3SA3,B3SAD218,CALL,0.03,0.10,18.51,21.81,17.83,6.31,0.07,-0.03,-42.86,9,2026-04-17,11.08,0.16,6.57,19.0
3138,BBAS3,BBASD264,CALL,0.05,0.10,23.43,25.77,9.99,9.56,0.09,-0.01,-11.11,9,2026-04-17,25.00,0.21,8.65,23.5


# A grande a aposta

In [14]:
df = df_final[df_final["tipo"] == "CALL"]

In [15]:
df[
    (df['tipo'] == 'CALL') &
    (df['ask'].between(0.01, 0.2)) &
    (df['volume'] > 0) &
    # (df['delta'].between(0.15, 0.45)) &  # alguma chance real
    (df['dias'] >= 0) &  # evita vencimento imediato
    ((df['strike'] / df['preco_atual']) <= 1.05)  # até 15% fora do dinheiro
][[
    'ativo','symbol','vencimento','dias','preco_atual',
    'strike','ask','bid','volume','delta','theta','poe'
]]

,ativo,symbol,vencimento,dias,preco_atual,strike,ask,bid,volume,delta,theta,poe
5332,BBSE3,BBSED359W2,2026-04-10,4,35.50,35.93,0.18,0.04,5000,0.338253,-0.043631,33.00
5422,BBSE3,BBSED390,2026-04-17,9,35.50,36.43,0.20,0.11,44400,0.277749,-0.028105,26.65
5426,BBSE3,BBSED371,2026-04-17,9,35.50,37.18,0.08,0.01,2300,0.116771,-0.015517,11.03
7978,ITSA4,ITSAD146,2026-04-17,9,13.93,14.57,0.18,0.05,62100,0.255737,-0.016414,23.75
